# 당뇨 환자 재입원 예측 — EDA와 모델 학습 (Diabetes 130)

- 데이터: 미국 130개 병원 당뇨 입원 기록 (101,766행 · 50열)
- 목표: 30일 내 재입원 여부 예측 — 이진분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_Diabetes130.txt

- 이 데이터의 특징: **대규모 실전 EMR**
  ('?' 결측 · 3범주 타깃 · 환자 중복 · 결측 과다 컬럼)

## 1. 불러오기 — '?' 결측

- 결측이 '?'로 표기됨 → na_values="?" 필수
- 컬럼 50개 · 혼합 타입 → low_memory=False

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("diabetic_data.csv", na_values="?",
                 low_memory=False)
print(df.shape)          # (101766, 50)
df.head()

## 2. 학습 전 확인

- 이 데이터는 사람이 손봐야 할 것이 넷

### 2-1. 타깃을 이진으로 변환

- readmitted 원본은 3범주: NO / >30(30일 후) / <30(30일 내)
- 목표가 '30일 내 재입원'이면 <30 만 1, 나머지 0
- 3범주 그대로보다 문제가 명확해짐

In [ ]:
print("원본 분포:", df["readmitted"].value_counts().to_dict())

df["readmit_30d"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])   # 원본 제거 (누수 방지)
print("이진 변환 후:", df["readmit_30d"].value_counts().to_dict())

### 2-2. 같은 환자가 여러 번 등장

- 환자 71,518명이 101,766행 (한 환자가 여러 번 입원)
- 무작위 분할 시 같은 환자가 train·test 양쪽에 → 과대평가
- 환자(patient_nbr) 단위로 분할

In [ ]:
print("환자 수:", df["patient_nbr"].nunique(), "/ 행 수:", len(df))

ids = df["patient_nbr"].unique()
rng = np.random.RandomState(42)
train_ids = rng.choice(ids, size=int(len(ids)*0.8), replace=False)
train_data = df[df["patient_nbr"].isin(train_ids)].copy()
test_data  = df[~df["patient_nbr"].isin(train_ids)].copy()
print("train:", train_data.shape, "test:", test_data.shape)

### 2-3. 식별자 컬럼 제거

- encounter_id(입원 번호), patient_nbr(환자 번호)는 식별자
- 예측에 무의미 → 제거

In [ ]:
for d in [train_data, test_data]:
    d.drop(columns=["encounter_id", "patient_nbr"], inplace=True)
print("식별자 제거 후:", train_data.shape)

### 2-4. 결측 과다 컬럼

- weight 97% · max_glu_serum 95% · A1Cresult 83% 결측
- weight는 거의 비어 정보 없음 → 제거 검토
- 검사(max_glu_serum 등)는 '미검사'라 삭제 대신 둬도 됨

In [ ]:
miss = train_data.isna().mean().sort_values(ascending=False)
print(miss.head(6).round(3))

# weight만 제거 (거의 전부 결측)
for d in [train_data, test_data]:
    d.drop(columns=["weight"], inplace=True)

## 3. EDA

- 대규모라 리포트 생성이 오래 걸림 → 표본 또는 minimal

In [ ]:
from data_profiling import ProfileReport

# 표본 1만 행으로 리포트 (전체는 무거움)
sample = train_data.sample(10000, random_state=42)
profile = ProfileReport(sample, minimal=True, progress_bar=False)
profile.to_file("diabetes_eda.html")   # 브라우저에서 열기

### 3-1. 재입원과 재원 기간

- time_in_hospital(재원 일수)이 길수록 재입원이 많은가

In [ ]:
import matplotlib.pyplot as plt
(train_data.groupby("readmit_30d")["time_in_hospital"]
   .mean().plot(kind="bar", figsize=(5,3),
                title="mean stay by readmission"))
plt.show()

## 4. 학습

- 불균형(재입원 약 11%) → roc_auc
- 결측·인코딩·범주 처리는 내부 자동 (컬럼 많아도 OK)

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label="readmit_30d",
    eval_metric="roc_auc",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

In [ ]:
print(predictor.evaluate(test_data))

### 5-1. 변수 중요도

- 무엇이 재입원을 예측하는가
- 재원 기간·진단 수·약물 변경 등이 상위에 오는가

In [ ]:
predictor.feature_importance(test_data).head(15)

## 정리

- '?' 결측 → na_values 필수
- 3범주 → 이진 타깃 변환 (문제 명확화)
- 환자 중복 → 환자 단위 분할 (누수 방지)
- 식별자·결측 과다 컬럼 제거
- 불균형 → roc_auc
- 학습·앙상블은 TabularPredictor가 자동

- 10만 행 대규모여도 흐름은 같다
  → 규모가 커도 사람의 판단(타깃 정의·분할)은 여전히 필요